In [ ]:
# This notebook is based on 10-fold-cv and SHAP analysis on Hanna daset for crisprAidit

In [ ]:
import pandas as pd     # Retraining model crisprAidit on 10-foldCV on Hanna's Datset(11088 rows)
import numpy as np
import os
import joblib
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr

# === 1. Load Dataset ===
data_path = r'C:\\Users\\faiza\\Downloads\\All excel files training testing\\Data sets for all tools\\63nt CRISPRAidit scores data - Copy - Hanna - Copy.xlsx'
data = pd.read_excel(data_path)
data = data.dropna(subset=['CRISPRAidit Score'])

# === 2. Prepare Off-Target Sequence ===
if 'off-target sequence' not in data.columns:
    data['off-target sequence'] = data['target sequence'].apply(lambda x: x[::-1])

# === 3. Feature Extraction ===
def one_hot_encode(seq, length=23):
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    encoded = np.zeros((length, 4))
    for i, char in enumerate(seq[:length]):
        if char in mapping:
            encoded[i, mapping[char]] = 1
    return encoded.flatten()

def compute_mismatch_features(seq1, seq2, length=23):
    mismatch_flags = np.zeros(length)
    nucleotide_pairs = np.zeros((length, 12))
    mismatch_map = {
        ('A', 'C'): 0, ('A', 'G'): 1, ('A', 'T'): 2,
        ('C', 'A'): 3, ('C', 'G'): 4, ('C', 'T'): 5,
        ('G', 'A'): 6, ('G', 'C'): 7, ('G', 'T'): 8,
        ('T', 'A'): 9, ('T', 'C'): 10, ('T', 'G'): 11
    }
    for i in range(length):
        if i < len(seq1) and i < len(seq2) and seq1[i] != seq2[i]:
            mismatch_flags[i] = 1
            pair = (seq1[i], seq2[i])
            if pair in mismatch_map:
                nucleotide_pairs[i, mismatch_map[pair]] = 1
    return np.concatenate([mismatch_flags, nucleotide_pairs.flatten()])

def pam_encoding(pam_seq):
    pam_dict = {'GG': [1, 0, 0, 0, 0, 0, 0, 0], 
                'AG': [0, 1, 0, 0, 0, 0, 0, 0],
                'GT': [0, 0, 1, 0, 0, 0, 0, 0],
                'GC': [0, 0, 0, 1, 0, 0, 0, 0],
                'GA': [0, 0, 0, 0, 1, 0, 0, 0],
                'TG': [0, 0, 0, 0, 0, 1, 0, 0],
                'CG': [0, 0, 0, 0, 0, 0, 1, 0],
                'other': [0, 0, 0, 0, 0, 0, 0, 1]}
    return pam_dict.get(pam_seq, pam_dict['other'])

def gc_content(seq):
    return (seq.count('G') + seq.count('C')) / len(seq)

def extract_all_features(df):
    features_list = []
    for _, row in df.iterrows():
        seq_features = one_hot_encode(row['target sequence'])
        mismatch_features = compute_mismatch_features(row['target sequence'], row['off-target sequence'])
        pam_features = np.array(pam_encoding(row.get('PAM', 'GG')))
        gc_feature = np.array([gc_content(row['target sequence'])])
        placeholder_features = np.zeros(473 - (len(seq_features) + len(mismatch_features) + len(pam_features) + len(gc_feature)))
        combined = np.concatenate([seq_features, mismatch_features, pam_features, gc_feature, placeholder_features])
        features_list.append(combined)
    return np.stack(features_list)

# === 4. Feature Names ===
sequence_feature_names = [f"{nucleotide}{i:02d}" for i in range(23) for nucleotide in "ACGT"]
mismatch_feature_names = (
    [f"M{i:02d}" for i in range(23)] +
    [f"{pair}{i:02d}" for i in range(23) for pair in ["AC", "AG", "AT", "CA", "CG", "CT", "GA", "GC", "GT", "TA", "TC", "TG"]]
)
pam_feature_names = ["PAM_GG", "PAM_AG", "PAM_GT", "PAM_GC", "PAM_GA", "PAM_TG", "PAM_CG", "PAM_other"]
gc_feature_name = ["GC_Content"]
placeholder_feature_names = [f"Placeholder_{i}" for i in range(473 - (
    len(sequence_feature_names) + len(mismatch_feature_names) + len(pam_feature_names) + len(gc_feature_name)
))]
all_feature_names = sequence_feature_names + mismatch_feature_names + pam_feature_names + gc_feature_name + placeholder_feature_names

# === 5. Extract features and target ===
X = extract_all_features(data)
y = data['CRISPRAidit Score'].values

# === 6. Cross-validation + training ===
kf = KFold(n_splits=10, shuffle=True, random_state=42)

all_predictions = []
pearson_scores = []
train_pearsons = []
r2_scores = []
rmse_scores = []
mse_scores = []

# Output base folder
base_dir = os.path.expanduser("~/Downloads/10_folds_CRISPRAidit")
os.makedirs(base_dir, exist_ok=True)

print("======== PER-FOLD METRICS =========")
for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    print(f"Training Fold {fold}...")
    fold_dir = os.path.join(base_dir, f"Fold{fold}")
    os.makedirs(fold_dir, exist_ok=True)

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = MLPRegressor(hidden_layer_sizes=(200, 100),
                         max_iter=1000,
                         alpha=0.001,
                         early_stopping=True,
                         validation_fraction=0.1,
                         n_iter_no_change=20,
                         random_state=42)
    
    model.fit(X_train_scaled, y_train)

    # Save model and scaler
    joblib.dump(model, os.path.join(fold_dir, f'CRISPRAidit_model_fold_{fold}.pkl'))
    joblib.dump(scaler, os.path.join(fold_dir, f'CRISPRAidit_scaler_fold_{fold}.pkl'))

    y_pred = model.predict(X_test_scaled)
    y_train_pred = model.predict(X_train_scaled)

    pearson = pearsonr(y_test, y_pred)[0]
    pearson_train = pearsonr(y_train, y_train_pred)[0]
    r2 = r2_score(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    mse = mean_squared_error(y_test, y_pred)

    pearson_scores.append(pearson)
    train_pearsons.append(pearson_train)
    r2_scores.append(r2)
    rmse_scores.append(rmse)
    mse_scores.append(mse)

    print(f"Fold {fold}: R2 = {r2:.4f}, RMSE = {rmse:.4f}, Pearson Train = {pearson_train:.4f}, Pearson Test = {pearson:.4f}")

    train_df = pd.DataFrame(X_train, columns=all_feature_names)
    train_df.insert(0, 'CRISPRAidit_GT', y_train)
    train_df.insert(1, 'CRISPRAidit_Prediction', y_train_pred)
    train_df.insert(2, 'Fold', fold)
    train_df.to_excel(os.path.join(fold_dir, f"train_fold{fold}.xlsx"), index=False)

    test_df = pd.DataFrame(X_test, columns=all_feature_names)
    test_df.insert(0, 'CRISPRAidit_GT', y_test)
    test_df.insert(1, 'CRISPRAidit_Prediction', y_pred)
    test_df.insert(2, 'Fold', fold)
    test_df.to_excel(os.path.join(fold_dir, f"test_fold{fold}.xlsx"), index=False)
    test_df.to_excel(os.path.join(fold_dir, f"test_predictions_fold{fold}.xlsx"), index=False)

    all_predictions.append(test_df)

# === 7. Save combined predictions ===
final_df = pd.concat(all_predictions, ignore_index=True)
final_path = os.path.join(base_dir, 'CRISPRAidit_10Fold_Predictions_WithExactFeatures.xlsx')
final_df.to_excel(final_path, index=False)

print("\n========= SUMMARY METRICS =========")
print(f"        Average R2 (Test) = {np.mean(r2_scores):.6f}")
print(f"               MSE (Test) = {np.mean(mse_scores):.6f}")
print(f"              RMSE (Test) = {np.mean(rmse_scores):.6f}")
print(f"           Pearson (Test) = {np.mean(pearson_scores):.6f}")
print(f"  Average Pearson (Train) = {np.mean(train_pearsons):.6f}")
print(f"\nFinal predictions saved to:\n  {final_path}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
import shap
import matplotlib.pyplot as plt
import joblib

# === 1. Load Dataset ===
data_path = r'C:\Users\faiza\Downloads\All excel files training testing\Data sets for all tools\63nt CRISPRAidit scores data - Copy - Hanna - Copy.xlsx'
data = pd.read_excel(data_path, engine='openpyxl')

# === 2. Ensure Off-Target Column Exists ===
if 'off-target sequence' not in data.columns:
    data['off-target sequence'] = data['target sequence'].apply(lambda x: x[::-1])

# === 3. Feature Extraction Functions ===
def one_hot_encode(seq, length=23):
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    encoded = np.zeros((length, 4))
    for i, char in enumerate(seq[:length]):
        if char in mapping:
            encoded[i, mapping[char]] = 1
    return encoded.flatten()

def compute_mismatch_features(seq1, seq2, length=23):
    mismatch_flags = np.zeros(length)
    nucleotide_pairs = np.zeros((length, 12))
    mismatch_map = {
        ('A', 'C'): 0, ('A', 'G'): 1, ('A', 'T'): 2,
        ('C', 'A'): 3, ('C', 'G'): 4, ('C', 'T'): 5,
        ('G', 'A'): 6, ('G', 'C'): 7, ('G', 'T'): 8,
        ('T', 'A'): 9, ('T', 'C'): 10, ('T', 'G'): 11
    }
    for i in range(length):
        if i < len(seq1) and i < len(seq2) and seq1[i] != seq2[i]:
            mismatch_flags[i] = 1
            pair = (seq1[i], seq2[i])
            if pair in mismatch_map:
                nucleotide_pairs[i, mismatch_map[pair]] = 1
    return np.concatenate([mismatch_flags, nucleotide_pairs.flatten()])

def pam_encoding(pam_seq):
    pam_dict = {'GG': [1, 0, 0, 0, 0, 0, 0, 0], 
                'AG': [0, 1, 0, 0, 0, 0, 0, 0],
                'GT': [0, 0, 1, 0, 0, 0, 0, 0],
                'GC': [0, 0, 0, 1, 0, 0, 0, 0],
                'GA': [0, 0, 0, 0, 1, 0, 0, 0],
                'TG': [0, 0, 0, 0, 0, 1, 0, 0],
                'CG': [0, 0, 0, 0, 0, 0, 1, 0],
                'other': [0, 0, 0, 0, 0, 0, 0, 1]}
    return pam_dict.get(pam_seq, pam_dict['other'])

def gc_content(seq):
    return (seq.count('G') + seq.count('C')) / len(seq)

def extract_all_features(data):
    features_list = []
    for idx, row in data.iterrows():
        try:
            seq_features = one_hot_encode(row['target sequence'], length=23)
            mismatch_features = compute_mismatch_features(row['target sequence'], row['off-target sequence'], length=23)
            pam_features = np.array(pam_encoding(row.get('PAM', 'GG')))
            gc_feature = np.array([gc_content(row['target sequence'])])
            placeholder_features = np.zeros(473 - (len(seq_features) + len(mismatch_features) + len(pam_features) + len(gc_feature)))
            combined_features = np.concatenate([seq_features, mismatch_features, pam_features, gc_feature, placeholder_features])
            if len(combined_features) != 473:
                raise ValueError(f"Unexpected feature length: {len(combined_features)}")
            features_list.append(combined_features)
        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            continue
    if not features_list:
        raise ValueError("No valid features extracted.")
    return np.stack(features_list)

# === 4. Feature Names ===
sequence_feature_names = [f"{nucleotide}{i:02d}" for i in range(23) for nucleotide in "ACGT"]
mismatch_feature_names = ([f"M{i:02d}" for i in range(23)] +
                          [f"{pair}{i:02d}" for i in range(23) for pair in ["AC", "AG", "AT", "CA", "CG", "CT", "GA", "GC", "GT", "TA", "TC", "TG"]])
pam_feature_names = ["PAM_GG", "PAM_AG", "PAM_GT", "PAM_GC", "PAM_GA", "PAM_TG", "PAM_CG", "PAM_other"]
gc_feature_name = ["GC_Content"]
placeholder_feature_names = [f"Placeholder_{i}" for i in range(473 - (len(sequence_feature_names) + len(mismatch_feature_names) + len(pam_feature_names) + len(gc_feature_name)))]

all_feature_names = sequence_feature_names + mismatch_feature_names + pam_feature_names + gc_feature_name + placeholder_feature_names

# === 5. Extract Features and Drop NaNs in Target ===
features = extract_all_features(data)
target = data['CRISPRAidit Score'].copy()

# Filter out rows with NaN target
features = features[~target.isna()]
target = target.dropna()

# Normalize
scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features)

# === 6. Train the Model ===
model = MLPRegressor(hidden_layer_sizes=(550, 490, 50, 50, 170), max_iter=330, alpha=0.008813467753305038, random_state=42)
model.fit(features_scaled, target)
predictions = model.predict(features_scaled)
data = data.loc[target.index]
data['Predictions'] = predictions

# === 7. SHAP Analysis ===
print("Calculating SHAP values...")
required_max_evals = 2 * features_scaled.shape[1] + 1
explainer = shap.Explainer(model.predict, features_scaled)
shap_values = explainer(features_scaled, max_evals=required_max_evals)
base_value_unscaled = predictions.mean()

# === 8. Save Everything to Excel ===
shap_columns = [f"SHAP_Value_{name}" for name in all_feature_names]
shap_df = pd.DataFrame(shap_values.values, columns=shap_columns)
feature_df = pd.DataFrame(features, columns=all_feature_names)
data = pd.concat([data.reset_index(drop=True), feature_df, shap_df], axis=1)

output_path = r'C:\Users\faiza\Downloads\predictions_shap_values_with_all_features.xlsx'
data.to_excel(output_path, index=False)
print(" Predictions, features, scores, and SHAP values saved to:", output_path)

# === 9. SHAP Waterfall Plot ===
idx = 0
shap_values_object = shap.Explanation(
    values=shap_values[idx].values,
    base_values=base_value_unscaled,
    data=features_scaled[idx],
    feature_names=all_feature_names
)
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values_object, max_display=10, show=True)
plt.title(f"SHAP Waterfall Plot for Row {idx}")
plt.savefig(r'C:\Users\faiza\Downloads\shap_waterfall_plot_row_0.png')
plt.close()
print(" SHAP waterfall plot saved for row 0.")

# === 10. Save Model and Scaler ===
model_path = r'C:\Users\faiza\Downloads\crispraidit_model.pkl'
scaler_path = r'C:\Users\faiza\Downloads\minmax_scaler.pkl'
joblib.dump(model, model_path)
joblib.dump(scaler, scaler_path)
print(" Model saved to:", model_path)
print(" Scaler saved to:", scaler_path)
